In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

# Verify versions
import langgraph
import langchain

### **Part 1: The Command Class — Combined State Update + Routing**

In [2]:
from typing_extensions import TypedDict
from typing import Literal
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command

class SupportState(TypedDict):
    message: str
    category: str
    priority: str
    response: str


def classify_ticket(state: SupportState) -> Command:
    """Classify support ticket and route — in a single step."""
    message = state["message"].lower()

    # Determine category
    if any(word in message for word in ["billing", "charge", "refund", "payment"]):
        category = "billing"
        priority = "high"
    elif any(word in message for word in ["bug", "error", "crash", "broken"]):
        category = "technical"
        priority = "high"
    elif any(word in message for word in ["feature", "request", "suggest"]):
        category = "feature_request"
        priority = "low"
    else:
        category = "general"
        priority = "medium"

    print(f"\U0001f3f7\ufe0f  Classified: {category} ({priority} priority)")

    return Command(
        update={"category": category, "priority": priority},
        goto=f"handle_{category}",
    )


def handle_billing(state: SupportState) -> dict:
    return {"response": f"\U0001f4b3 Billing team will review: '{state['message'][:50]}...'"}

def handle_technical(state: SupportState) -> dict:
    return {"response": f"\U0001f527 Engineering team assigned: '{state['message'][:50]}...'"}

def handle_feature_request(state: SupportState) -> dict:
    return {"response": f"\U0001f4a1 Added to product backlog: '{state['message'][:50]}...'"}

def handle_general(state: SupportState) -> dict:
    return {"response": f"\U0001f4e7 General support team: '{state['message'][:50]}...'"}

# Build graph — no add_conditional_edges needed!
builder = StateGraph(SupportState)
builder.add_node("classify", classify_ticket)
builder.add_node("handle_billing", handle_billing)
builder.add_node("handle_technical", handle_technical)
builder.add_node("handle_feature_request", handle_feature_request)
builder.add_node("handle_general", handle_general)

builder.add_edge(START, "classify")
for handler in ["handle_billing", "handle_technical", "handle_feature_request", "handle_general"]:
    builder.add_edge(handler, END)

app = builder.compile()

# Test various ticket types
test_tickets = [
    "I was charged twice for my subscription last month",
    "The app crashes when I try to export PDF files",
    "Can you add dark mode to the dashboard?",
    "How do I update my email address?",
]

for ticket in test_tickets:
    result = app.invoke({"message": ticket, "category": "", "priority": "", "response": ""})
    print(f"\u2192 {result['response']}\n")



🏷️  Classified: billing (high priority)
→ 💳 Billing team will review: 'I was charged twice for my subscription last month...'

🏷️  Classified: technical (high priority)
→ 🔧 Engineering team assigned: 'The app crashes when I try to export PDF files...'

🏷️  Classified: general (medium priority)
→ 📧 General support team: 'Can you add dark mode to the dashboard?...'

🏷️  Classified: general (medium priority)
→ 📧 General support team: 'How do I update my email address?...'



### **Command for HITL Resumption**

In [3]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from typing_extensions import TypedDict

class ReviewState(TypedDict):
    content: str
    reviewer: str
    approved: bool
    final_content: str

def generate_content(state: ReviewState) -> dict:
    content = f"Generated content based on requirements: {state['content']}"
    print(f"\u270d\ufe0f  Generated: {content}")
    return {"content": content}

def review_gate(state: ReviewState) -> Command:
    """Ask for human review — pause and wait."""
    decision = interrupt({
        "message": f"Review this content:\n\n{state['content']}\n\nApprove?",
        "options": ["approve", "reject", "request_changes"],
    })
    
    if decision == "approve":
        return Command(
            update={"approved": True},
            goto="publish",
        )
    elif decision == "request_changes":
        return Command(
            update={"approved": False},
            goto="generate_content",  # Loop back!
        )
    else:
        return Command(
            update={"approved": False},
            goto="cancel",
        )


def publish(state: ReviewState) -> dict:
    print(f"\U0001f680 Published: {state['content']}")
    return {"final_content": state["content"]}

def cancel(state: ReviewState) -> dict:
    print("\u274c Content rejected")
    return {"final_content": "REJECTED"}

db = sqlite3.connect(":memory:",check_same_thread=False)
memory = SqliteSaver(db)

builder = StateGraph(ReviewState)
builder.add_node("generate_content", generate_content)
builder.add_node("review_gate", review_gate)
builder.add_node("publish", publish)
builder.add_node("cancel", cancel)

builder.add_edge(START, "generate_content")
builder.add_edge("generate_content", "review_gate")
builder.add_edge("publish", END)
builder.add_edge("cancel", END)

app = builder.compile(checkpointer=memory)
config = {"configurable": {"thread_id": "content-001"}}

In [4]:
print("=== Running to review gate ===")
result = app.invoke(
    {"content": "AI safety best practices", "reviewer": "", "approved": False, "final_content": ""},
    config
)

=== Running to review gate ===
✍️  Generated: Generated content based on requirements: AI safety best practices


In [5]:
# Approve the content
print("\n=== Reviewer approves ===")
final = app.invoke(Command(resume="approve"), config)
print(f"Final: {final['final_content']}")


=== Reviewer approves ===
🚀 Published: Generated content based on requirements: AI safety best practices
Final: Generated content based on requirements: AI safety best practices


### **Part 2: The Functional API — @entrypoint and @task**

In [6]:
import asyncio
from langgraph.func import entrypoint, task
from langgraph.checkpoint.memory import MemorySaver
from langchain_openai import AzureChatOpenAI

llm = AzureChatOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
    temperature=0.7,
)

@task
def research_topic(topic: str) -> str:
    """Research a topic using the LLM."""
    print(f"\U0001f52c Researching: {topic}")
    response = llm.invoke(f"Give me 2 key facts about: {topic}. Be concise.")
    return response.content

@task
def summarize_findings(findings: list) -> str:
    """Summarize multiple research findings."""
    print(f"\U0001f4ca Summarizing {len(findings)} findings...")
    combined = "\n".join(f"- {f}" for f in findings)
    response = llm.invoke(f"Summarize these findings in 2 sentences:\n{combined}")
    return response.content

@entrypoint(checkpointer=MemorySaver())
def research_workflow(query: str) -> dict:
    """Research multiple related topics and synthesize findings.

    Standard Python control flow — no graph construction needed!
    """
    # Determine sub-topics (could also call another @task here)
    topics = [f"{query} overview", f"{query} applications", f"{query} challenges"]

    # Launch parallel research tasks (lazy — .result() waits for completion)
    research_tasks = [research_topic(topic) for topic in topics]

    # Wait for all results
    findings = [t.result() for t in research_tasks]

    # Summarize
    summary = summarize_findings(findings).result()

    return {
        "query": query,
        "findings": findings,
        "summary": summary
    }

# Call it like a regular function
print("=== Functional API Research Workflow ===")
result = research_workflow.invoke(
    "quantum computing",
    config={"configurable": {"thread_id": "session-1"}}
)
print(f"\nQuery: {result['query']}")
print(f"\nSummary: {result['summary']}")

=== Functional API Research Workflow ===
🔬 Researching: quantum computing overview
🔬 Researching: quantum computing applications
🔬 Researching: quantum computing challenges
📊 Summarizing 3 findings...

Query: quantum computing

Summary: Quantum computing harnesses principles like superposition and entanglement to process complex problems exponentially faster than classical computers, with promising applications in optimization, cryptography, and material science. However, challenges such as high error rates, decoherence, and scalability limitations hinder the stability and practicality of large-scale quantum systems.


### **Interoperability: Functional API and StateGraph**

In [7]:
from langgraph.func import entrypoint, task
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage

@task
def run_subgraph(input_text: str) -> str:
    """Run a StateGraph as a task inside an entrypoint."""
    # Build a mini StateGraph
    def process(state):
        return {"messages": [{"role": "assistant", "content": f"Processed: {input_text}"}]}

    builder = StateGraph(MessagesState)
    builder.add_node("process", process)
    builder.add_edge(START, "process")
    builder.add_edge("process", END)
    graph = builder.compile()

    result = graph.invoke({"messages": [HumanMessage(input_text)]})
    return result["messages"][-1].content


@entrypoint(checkpointer=MemorySaver())
def mixed_workflow(inputs: list) -> list:
    """Functional API calling StateGraph as tasks."""
    tasks = [run_subgraph(inp) for inp in inputs]
    return [t.result() for t in tasks]

result = mixed_workflow.invoke(
    ["Hello", "World", "LangGraph"],
    config={"configurable": {"thread_id": "mixed-001"}}
)
for r in result:
    print(r)

Processed: Hello
Processed: World
Processed: LangGraph


### **Part 3: Long-Term Memory with Stores**

In [8]:
from langgraph.store.memory import InMemoryStore
from langgraph.store.base import BaseStore
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import StateGraph, MessagesState, START, END
from langchain_openai import AzureChatOpenAI
from langchain_core.messages import HumanMessage

db = sqlite3.connect(":memory:",check_same_thread=False)
checkpointer = SqliteSaver(db)
store = InMemoryStore()

llm = AzureChatOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
    temperature=0.7,
)

def smart_assistant(state: MessagesState, store: BaseStore) -> dict:
    """Assistant that learns and remembers user preferences across conversations."""
    # Get user identity (in production, from auth context or config)
    user_id = "alice"

    # Read long-term memories
    user_memories = store.search(("users", user_id, "memories"))

    memory_context = ""
    if user_memories:
        facts = [item.value.get("content", "") for item in user_memories]
        memory_context = "\n\nWhat I know about you:\n" + "\n".join(f"\u2022 {f}" for f in facts)

    last_message = state["messages"][-1].content

    facts_to_store = []
    if "i prefer" in last_message.lower() or "i like" in last_message.lower():
        facts_to_store.append(last_message)
    if "my name is" in last_message.lower():
        name = last_message.lower().split("my name is")[-1].strip().split()[0]
        facts_to_store.append(f"User's name is {name}")

    for i, fact in enumerate(facts_to_store):
        key = f"fact_{len(user_memories) + i}"
        store.put(("users", user_id, "memories"), key, {"content": fact})
        print(f"\U0001f4be Stored to long-term memory: {fact}")


    # Generate response with memory context
    system_prompt = f"You are a helpful, personalized assistant.{memory_context}"
    response = llm.invoke(
        [{"role": "system", "content": system_prompt}] + state["messages"]
    )

    return {"messages": [response]}

# Build graph with BOTH checkpointer and store
builder = StateGraph(MessagesState)
builder.add_node("assistant", smart_assistant)
builder.add_edge(START, "assistant")
builder.add_edge("assistant", END)

app = builder.compile(checkpointer=checkpointer, store=store)

# Session 1 — user introduces themselves
print("=== Session 1 ===")
config1 = {"configurable": {"thread_id": "chat-001"}}
r = app.invoke({"messages": [HumanMessage("Hi! My name is Alice. I prefer Python over JavaScript.")]}, config1)
print("Bot:", r["messages"][-1].content)



=== Session 1 ===
💾 Stored to long-term memory: Hi! My name is Alice. I prefer Python over JavaScript.
💾 Stored to long-term memory: User's name is alice.
Bot: Hi Alice! It's great to meet you. Python is an awesome language—clean syntax, tons of libraries, and it's super beginner-friendly, yet powerful for advanced tasks. What kind of projects or coding challenges are you working on (or thinking about tackling) in Python? 😊


In [9]:
# Session 2 — DIFFERENT thread, but store remembers Alice
print("\n=== Session 2 (new thread — store still remembers!) ===")
config2 = {"configurable": {"thread_id": "chat-002"}}
r = app.invoke({"messages": [HumanMessage("Hey, what do you know about me?")]}, config2)
print("Bot:", r["messages"][-1].content)


=== Session 2 (new thread — store still remembers!) ===
Bot: Hi, Alice! I know that your name is Alice, and you prefer Python over JavaScript. 😊 Is there anything else you'd like me to remember about you?
